# Benchmark de Inferencia y Eficiencia Computacional - YOLO-Seg

Mide latencia y throughput de **YOLOv11-Seg** con el mismo protocolo que `benchmark_pidnet.ipynb`, para que los números sean comparables en la Tabla II del informe.

Se reportan dos lecturas, análogas a las de PIDNet:

1. **Forward puro** del `nn.Module` interno (`SegmentationModel`), medido con `torch.cuda.Event` + `synchronize()`, warm-up y N iteraciones. FP32 y AMP FP16.
2. **Pipeline end-to-end** vía `model.predict()` de Ultralytics: incluye preprocesamiento, NMS y ensamblado de máscaras (el ciclo real de inferencia).

Más el análisis de escalabilidad por batch y la fila LaTeX para la Tabla II.

> Correr en la **misma GPU que PIDNet** (Colab Tesla T4) para que la comparación sea válida.

In [ ]:
# ==== Solo para ejecutar en Colab ========
try:
    from google.colab import drive
    import os, sys
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/tp_computer_vision_ii'
    notebook_dir = os.path.join(project_path, 'src', 'yolo_seg')
    os.chdir(notebook_dir)
    if notebook_dir not in sys.path:
        sys.path.append(notebook_dir)
    print('Directorio actual en Colab:', os.getcwd())
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Ejecutando en entorno local:', __import__('os').getcwd())

## 1. Parámetros del benchmark

In [ ]:
from pathlib import Path

# Modelos a medir: nombre -> (run en disco, imgsz de inferencia)
# imgsz debe coincidir con el usado al entrenar cada variante.
MODELS = {
    'YOLO11n-Seg':  dict(run='yolo11n_seg_nano',  imgsz=512),
    'YOLO11s-Seg':  dict(run='yolo11s_seg_small', imgsz=512),
    'YOLO11n-Seg*': dict(run='yolo11n_seg_hires', imgsz=768),  # Hi-Res, fuera de la comparación estricta
}

WARMUP_ITERS       = 30    # iteraciones descartadas (mismo valor que PIDNet)
BENCHMARK_ITERS    = 150   # iteraciones de medición
BATCH_SIZES_TO_TEST = [1, 2, 4, 8]

SEG_DIR = Path.cwd() / 'runs' / 'yolo_seg'

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
if device.type == 'cuda':
    print('GPU        :', torch.cuda.get_device_name(0))
    print('VRAM       : %.2f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

## 2. Protocolo de medición

`bench_forward` mide el forward puro del `nn.Module` interno con `cuda.Event` (idéntico al de PIDNet). `bench_predict` mide el `predict()` completo de Ultralytics sobre un frame en memoria (sin lectura de disco).

In [ ]:
def _summarize(latencies):
    lat = np.array(latencies)
    return {
        'mean': float(np.mean(lat)), 'median': float(np.median(lat)),
        'std': float(np.std(lat)), 'p95': float(np.percentile(lat, 95)),
        'p99': float(np.percentile(lat, 99)), 'min': float(lat.min()), 'max': float(lat.max()),
        'fps': 1000.0 / float(np.mean(lat)), 'raw': lat,
    }


def bench_forward(net, input_tensor, device, warmup=30, iters=150, use_amp=False):
    """Forward puro del nn.Module interno de YOLO. Mismo protocolo que PIDNet."""
    net.eval()
    amp = (use_amp and device.type == 'cuda')
    with torch.inference_mode(), torch.amp.autocast('cuda', enabled=amp):
        for _ in range(warmup):
            _ = net(input_tensor)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    latencies = []
    with torch.inference_mode(), torch.amp.autocast('cuda', enabled=amp):
        for _ in range(iters):
            if device.type == 'cuda':
                s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                s.record(); _ = net(input_tensor); e.record()
                torch.cuda.synchronize()
                latencies.append(s.elapsed_time(e))
            else:
                t0 = time.perf_counter(); _ = net(input_tensor); latencies.append((time.perf_counter() - t0) * 1000)
    return _summarize(latencies)


def bench_predict(model, frame, imgsz, conf, device, warmup=30, iters=150):
    """predict() end-to-end de Ultralytics (preproc + forward + NMS + máscaras) sobre un frame en memoria."""
    for _ in range(warmup):
        _ = model.predict(frame, imgsz=imgsz, conf=conf, retina_masks=True, verbose=False)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    latencies = []
    for _ in range(iters):
        if device.type == 'cuda':
            torch.cuda.synchronize(); t0 = time.perf_counter()
            _ = model.predict(frame, imgsz=imgsz, conf=conf, retina_masks=True, verbose=False)
            torch.cuda.synchronize(); latencies.append((time.perf_counter() - t0) * 1000)
        else:
            t0 = time.perf_counter()
            _ = model.predict(frame, imgsz=imgsz, conf=conf, retina_masks=True, verbose=False)
            latencies.append((time.perf_counter() - t0) * 1000)
    return _summarize(latencies)

## 3. Forward puro (FP32 y AMP FP16) por modelo

In [ ]:
results = {}   # nombre -> dict con stats

for name, cfg in MODELS.items():
    w = SEG_DIR / cfg['run'] / 'weights' / 'best.pt'
    if not w.exists():
        print(f'[skip] {name}: faltan pesos ({w})'); continue
    yolo = YOLO(str(w))
    net = yolo.model.to(device).eval()
    imgsz = cfg['imgsz']
    n_params = sum(p.numel() for p in net.parameters()) / 1e6
    dummy = torch.randn(1, 3, imgsz, imgsz, device=device)

    fp32 = bench_forward(net, dummy, device, WARMUP_ITERS, BENCHMARK_ITERS, use_amp=False)
    fp16 = bench_forward(net, dummy, device, WARMUP_ITERS, BENCHMARK_ITERS, use_amp=True) if device.type == 'cuda' else None

    results[name] = {'yolo': yolo, 'imgsz': imgsz, 'params': n_params, 'fp32': fp32, 'fp16': fp16}
    print(f'\n=== {name}  (imgsz={imgsz}, {n_params:.2f}M params) ===')
    print(f'  Forward FP32 : {fp32["mean"]:.2f} ms +/- {fp32["std"]:.2f}  (mediana {fp32["median"]:.2f}, p95 {fp32["p95"]:.2f}) -> {fp32["fps"]:.2f} FPS')
    if fp16:
        print(f'  Forward FP16 : {fp16["mean"]:.2f} ms +/- {fp16["std"]:.2f} -> {fp16["fps"]:.2f} FPS  (speedup {fp32["mean"]/fp16["mean"]:.2f}x)')

## 4. Pipeline end-to-end (`predict` de Ultralytics)

Incluye preprocesamiento, forward, NMS y ensamblado de máscaras. Es el análogo al pipeline E2E de PIDNet. `conf` es el mejor umbral de cada modelo según `03_eval.ipynb`.

In [ ]:
# mejor conf por modelo (de 03_eval.ipynb). Ajustar si cambia el barrido.
BEST_CONF = {'YOLO11n-Seg': 0.01, 'YOLO11s-Seg': 0.01, 'YOLO11n-Seg*': 0.05}

for name, r in results.items():
    imgsz = r['imgsz']
    frame = np.random.randint(0, 255, (imgsz, imgsz, 3), dtype=np.uint8)  # frame en memoria, sin I/O
    e2e = bench_predict(r['yolo'], frame, imgsz, BEST_CONF.get(name, 0.05), device, WARMUP_ITERS, BENCHMARK_ITERS)
    r['e2e'] = e2e
    print(f'{name:14s}  pipeline E2E: {e2e["mean"]:.2f} ms +/- {e2e["std"]:.2f}  -> {e2e["fps"]:.2f} FPS')

## 5. Escalabilidad por batch (forward puro, FP16)

In [ ]:
for name, r in results.items():
    imgsz = r['imgsz']
    net = r['yolo'].model.to(device).eval()
    print(f'\n=== {name} (imgsz={imgsz}) ===')
    print(f'{"batch":<7}{"lat batch (ms)":<16}{"lat/img (ms)":<15}{"FPS":<10}')
    r['batch'] = []
    for bs in BATCH_SIZES_TO_TEST:
        xb = torch.randn(bs, 3, imgsz, imgsz, device=device)
        sb = bench_forward(net, xb, device, warmup=15, iters=50, use_amp=(device.type=='cuda'))
        per_img = sb['mean'] / bs
        fps = bs * 1000.0 / sb['mean']
        r['batch'].append({'bs': bs, 'lat': sb['mean'], 'per_img': per_img, 'fps': fps})
        print(f'{bs:<7}{sb["mean"]:<16.2f}{per_img:<15.2f}{fps:<10.2f}')

## 6. Gráficos y fila LaTeX para la Tabla II

Para la columna de latencia del informe se reporta el **forward FP32** (la misma lectura que la fila de PIDNet usa como latencia principal). El pipeline E2E y el batch quedan como dato complementario.

In [ ]:
if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    colors = ['royalblue', 'seagreen', 'crimson']
    for (name, r), c in zip(results.items(), colors):
        axes[0].hist(r['fp32']['raw'], bins=25, alpha=0.55, label=f'{name} ({r["fp32"]["mean"]:.1f} ms)', color=c, edgecolor='black')
    axes[0].set_xlabel('Latencia forward FP32 (ms)'); axes[0].set_ylabel('Iteraciones')
    axes[0].set_title('Distribución de latencia (YOLO-Seg)', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

    for (name, r), c in zip(results.items(), colors):
        bs = [b['bs'] for b in r['batch']]; fps = [b['fps'] for b in r['batch']]
        axes[1].plot(bs, fps, marker='o', lw=2, label=name, color=c)
    axes[1].set_xlabel('Batch size'); axes[1].set_ylabel('Throughput (FPS)')
    axes[1].set_title('Escalabilidad vs batch', fontweight='bold'); axes[1].set_xticks(BATCH_SIZES_TO_TEST)
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    Path('figures').mkdir(exist_ok=True)
    fig.savefig('figures/benchmark_yolo.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Fila LaTeX para la Tabla II (main.tex). Completar mIoU/IoU/F1/Prec/Recall desde 03_eval.
# métricas de calidad ya calculadas en 03_eval (comparison_final_eval.csv):
QUALITY = {
    'YOLO11s-Seg':  dict(miou=0.8295, iou=0.6756, f1=0.8064, prec=0.8450, rec=0.7711, variant='Small (imgsz=512)', bold=True),
    'YOLO11n-Seg':  dict(miou=0.8259, iou=0.6686, f1=0.8014, prec=0.8526, rec=0.7560, variant='Nano (imgsz=512)', bold=True),
    'YOLO11n-Seg*': dict(miou=0.8438, iou=0.7026, f1=0.8254, prec=0.8580, rec=0.7951, variant='Nano Hi-Res (imgsz=768)', bold=False),
}

print('=' * 90)
print('  FILAS PARA LA TABLA II (main.tex) - latencia = forward FP32, FPS correspondiente')
print('=' * 90)
for name, r in results.items():
    q = QUALITY.get(name)
    if q is None:
        continue
    lat, fps = r['fp32']['mean'], r['fp32']['fps']
    nm = f'\\textbf{{{name}}}' if q['bold'] else name
    print(f"{nm} & {q['variant']} & {q['miou']:.4f} & {q['iou']:.4f} & {q['f1']:.4f} & {q['prec']:.4f} & {q['rec']:.4f} & {lat:.2f} & {fps:.2f} \\\\")
print('=' * 90)
print('Nota: verificar que estos números salgan de la MISMA GPU (T4) que PIDNet antes de pegarlos.')